# EDSS 취업 코호트 기초 추세 검산

최종 학교–코호트 마트를 제한 DuckDB에서 다시 집계하고, 비교 구간 단절·진학자수 품질·커밋된 추세표를 검산한다. 파생 비율은 공식 취업률이 아니다.

In [1]:
import csv
import hashlib
import importlib.util
import json
from pathlib import Path

import duckdb
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SCRIPT = REPO_ROOT / 'scripts' / 'build_edss_employment_cohort_trends.py'
SPEC = importlib.util.spec_from_file_location('cohort_trends', SCRIPT)
cohort_trends = importlib.util.module_from_spec(SPEC)
SPEC.loader.exec_module(cohort_trends)
DB = REPO_ROOT / 'data/processed/edss/restricted/edss_all.duckdb'
CSV_PATH = REPO_ROOT / 'data/metadata/edss_employment_cohort_trends.csv'
JSON_PATH = REPO_ROOT / 'data/metadata/edss_employment_cohort_trends.json'

In [2]:
connection = duckdb.connect(str(DB), read_only=True)
try:
    recomputed = cohort_trends.build_trend_rows(
        cohort_trends.query_aggregates(connection)
    )
finally:
    connection.close()

by_year = {row['employment_cohort_year']: row for row in recomputed}
assert len(recomputed) == 11
assert by_year['2010']['reported_employed_share_change_pp_from_previous_comparable_cohort'] is None
assert by_year['2014']['reported_employed_share_change_pp_from_previous_comparable_cohort'] is None
assert all(by_year[year]['reported_further_study_share_of_graduates'] is None for year in ('2015', '2016', '2017', '2018'))
{'status': 'passed', 'cohort_count': len(recomputed), 'school_cohort_key_count': sum(row['school_count'] for row in recomputed)}

{'status': 'passed', 'cohort_count': 11, 'school_cohort_key_count': 5969}

In [3]:
lines = [
    '| 코호트 | 기준일 | 졸업학생 수 | 취업자 수 | 설명적 비율 | 직전 비교가능 코호트 대비 |',
    '|---:|---|---:|---:|---:|---:|',
]
for row in recomputed:
    change = row['reported_employed_share_change_pp_from_previous_comparable_cohort']
    change_text = '—' if change is None else f'{change:+.2f}%p'
    lines.append(
        f"| {row['employment_cohort_year']} | {row['employment_reference_date']} | "
        f"{row['reported_graduate_count']:,} | {row['reported_employed_count']:,} | "
        f"{row['reported_employed_share_of_graduates']:.2%} | {change_text} |"
    )
display(Markdown('\n'.join(lines)))

| 코호트 | 기준일 | 졸업학생 수 | 취업자 수 | 설명적 비율 | 직전 비교가능 코호트 대비 |
|---:|---|---:|---:|---:|---:|
| 2010 | 2010-06-01 | 539,071 | 267,003 | 49.53% | — |
| 2011 | 2011-06-01 | 559,000 | 292,028 | 52.24% | +2.71%p |
| 2012 | 2012-06-01 | 566,374 | 294,884 | 52.07% | -0.18%p |
| 2013 | 2013-06-01 | 555,141 | 284,660 | 51.28% | -0.79%p |
| 2014 | 2014-12-31 | 557,234 | 302,280 | 54.25% | — |
| 2015 | 2015-12-31 | 576,023 | 315,412 | 54.76% | +0.51%p |
| 2016 | 2016-12-31 | 580,695 | 318,438 | 54.84% | +0.08%p |
| 2017 | 2017-12-31 | 574,009 | 305,263 | 53.18% | -1.66%p |
| 2018 | 2018-12-31 | 555,808 | 299,883 | 53.95% | +0.77%p |
| 2019 | 2019-12-31 | 550,354 | 291,210 | 52.91% | -1.04%p |
| 2020 | 2020-12-31 | 553,521 | 283,694 | 51.25% | -1.66%p |

In [4]:
with CSV_PATH.open(encoding='utf-8', newline='') as handle:
    committed = list(csv.DictReader(handle))
summary = json.loads(JSON_PATH.read_text(encoding='utf-8'))
csv_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
assert len(committed) == len(recomputed) == summary['output']['row_count']
assert tuple(row['employment_cohort_year'] for row in committed) == cohort_trends.EXPECTED_COHORTS
assert committed[4]['previous_comparable_cohort_year'] == ''
assert committed[4]['reported_employed_share_change_pp_from_previous_comparable_cohort'] == ''
assert csv_sha256 == summary['output']['sha256']
{'status': 'passed', 'csv_sha256': csv_sha256, 'guardrail_count': len(summary['guardrails'])}

{'status': 'passed',
 'csv_sha256': '6d98f884a6487d55f347113a51f665b68bc7a1524cc553229485ef3ee424d961',
 'guardrail_count': 4}